In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# ==========================
# 1️⃣ 数据加载器定义
# ==========================

In [2]:
dataset_base_path = r"E:\workspace\python_work\dataSet\LLVIP"


class LLVIPDataset(Dataset):
    def __init__(self, root_ir, root_vi, transform=None):
        # 排序确保对应关系
        self.ir_paths = sorted([os.path.join(root_ir, f) for f in os.listdir(root_ir)])
        self.vi_paths = sorted([os.path.join(root_vi, f) for f in os.listdir(root_vi)])
        self.transform = transform

    def __len__(self):
        return len(self.ir_paths)

    def __getitem__(self, idx):
        ir = Image.open(self.ir_paths[idx]).convert("L")  # 灰度
        vi = Image.open(self.vi_paths[idx]).convert("L")
        if self.transform:
            ir = self.transform(ir)
            vi = self.transform(vi)
        return ir, vi



# ==========================
# 2️⃣ 模型结构（共享结构）
# ==========================

In [3]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 2, 1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

# ==========================
# 3️⃣ 数据预处理与加载
# ==========================

In [4]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_ir_path = os.path.join(dataset_base_path, "infrared", "train")
train_vi_path = os.path.join(dataset_base_path, "visible", "train")

trainset = LLVIPDataset(
    root_ir=train_ir_path,
    root_vi=train_vi_path,
    transform=transform
)
# 加载训练集
trainloader = DataLoader(trainset, batch_size=8, shuffle=True)


# ==========================
# 4️⃣ 模型初始化与优化器
# ==========================

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
AE_ir = AutoEncoder().to(device)
AE_vi = AutoEncoder().to(device)

optimizer_ir = optim.Adam(AE_ir.parameters(), lr=1e-3)
optimizer_vi = optim.Adam(AE_vi.parameters(), lr=1e-3)
criterion = nn.L1Loss()  # 可以改成 SSIM+L1
device

'cuda'

# ==========================
# 5️⃣ 训练循环
# ==========================

In [6]:
epochs = 20
# 创建保存模型的目录
os.makedirs("weights", exist_ok=True)
os.makedirs("weights/checkpoints", exist_ok=True)  # 创建检查点目录

for epoch in range(epochs):
    AE_ir.train(); AE_vi.train()
    total_loss_ir, total_loss_vi = 0, 0

    for ir, vi in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}"):
        ir, vi = ir.to(device), vi.to(device)

        # IR 自编码器
        recon_ir = AE_ir(ir)
        loss_ir = criterion(recon_ir, ir)
        optimizer_ir.zero_grad()
        loss_ir.backward()
        optimizer_ir.step()
        total_loss_ir += loss_ir.item()

        # VI 自编码器
        recon_vi = AE_vi(vi)
        loss_vi = criterion(recon_vi, vi)
        optimizer_vi.zero_grad()
        loss_vi.backward()
        optimizer_vi.step()
        total_loss_vi += loss_vi.item()
    # 计算平均损失
    avg_loss_ir = total_loss_ir / len(trainloader)
    avg_loss_vi = total_loss_vi / len(trainloader)
    print(f"Epoch {epoch + 1}/{epochs} |AVG IR loss: {avg_loss_ir:.4f} | AVG VI loss: {avg_loss_vi:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | IR loss: {total_loss_ir/len(trainloader):.4f} | VI loss: {total_loss_vi/len(trainloader):.4f}")

    # ==========================
    # 每个epoch结束后保存模型检查点
    # ==========================
    # 保存当前epoch的模型
    torch.save({
        'epoch': epoch + 1,
        'AE_ir_state_dict': AE_ir.state_dict(),
        'AE_vi_state_dict': AE_vi.state_dict(),
        'optimizer_ir_state_dict': optimizer_ir.state_dict(),
        'optimizer_vi_state_dict': optimizer_vi.state_dict(),
        'loss_ir': avg_loss_ir,
        'loss_vi': avg_loss_vi
    }, f"weights/checkpoints/epoch_{epoch+1:02d}.pth")

Epoch 1/20:   2%|▏         | 33/1504 [00:27<20:29,  1.20it/s]


KeyboardInterrupt: 

# ==========================
# 6️⃣ 保存模型
# ==========================

In [ ]:

torch.save(AE_ir.state_dict(), "weights/AE_IR_final.pth")
torch.save(AE_vi.state_dict(), "weights/AE_VI_final.pth")

# ==========================
# 7️⃣ 可视化样例
# ==========================

In [ ]:

AE_ir.eval()
ir, vi = next(iter(trainloader))
with torch.no_grad():
    recon_ir = AE_ir(ir.to(device)).cpu()
    recon_vi = AE_vi(vi.to(device)).cpu()

def show_pair(original, recon, title):
    plt.figure(figsize=(8,3))
    for i in range(4):
        plt.subplot(2,4,i+1)
        plt.imshow(original[i][0], cmap='gray'); plt.axis('off')
        if i==0: plt.title(f'{title} Original')
        plt.subplot(2,4,i+5)
        plt.imshow(recon[i][0], cmap='gray'); plt.axis('off')
        if i==0: plt.title(f'{title} Reconstructed')
    plt.show()

show_pair(ir, recon_ir, "IR")
show_pair(vi, recon_vi, "VI")

In [ ]:
import numpy
print("✅ numpy version:", np.__version__)

In [ ]:
import sys
import torch
import numpy as np

print("=== 环境诊断 ===")
print(f"Python版本: {sys.version}")
print(f"PyTorch版本: {torch.__version__}")
print(f"NumPy版本: {np.__version__}")

# 检查CUDA支持
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")

# 测试NumPy和PyTorch兼容性
try:
    # 测试NumPy数组转换为PyTorch张量
    np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
    torch_tensor = torch.from_numpy(np_array)
    print("✅ NumPy到PyTorch转换测试通过")
except Exception as e:
    print(f"❌ 转换测试失败: {e}")

# 测试ToTensor转换
try:
    from torchvision import transforms
    transform = transforms.ToTensor()
    print("✅ torchvision transforms导入成功")
except Exception as e:
    print(f"❌ torchvision导入失败: {e}")